In [ ]:
# Mount Google Drive to access input data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Install required geospatial library
!pip install rioxarray


In [ ]:
# ============================================================
# Imports and Configuration
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import xarray as xr
import rioxarray
import numpy as np
from tqdm import tqdm
from scipy import stats

# --- File paths ---
# ECOSTRESS mean LST composite (May–Nov 2023)
ECO_PATH = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST/cloudmasked_output/Composites/<YOUR_COMPOSITE>.tif"

# NOAA CRW SST NetCDF (same time period)
CRW_PATH = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/NOAA CRW/Data/<YOUR_CRW_FILE>.nc"

# Max bleaching per survey location (Jun–Dec 2023)
BLEACHING_CSV_PATH = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/NOAA CRW/Data/max_bleaching_surveys_per_location_Jun-Dec2023.csv"

# --- Column names ---
LAT_COL   = 'obs_lat'
LON_COL   = 'obs_lon'
BLEACH_COL = 'Bleached_Percent'

# --- Analysis parameters ---
# Temperature threshold (°C) used to split sites into low- and high-stress groups
# for Figures 5a and 5b. Set based on quantile exploration of the data.
LST_THRESHOLD = 31.3

# Target CRS for all spatial data (GCS North American 1983)
TARGET_CRS = "EPSG:4269"

# Time window for CRW SST temporal average
CRW_TIME_START = "2023-05-01"
CRW_TIME_END   = "2023-11-30"


In [ ]:
# ============================================================
# Load Data and Define Spatial Sampling Function
# ============================================================
# Loads the bleaching CSV, the ECOSTRESS LST raster, and
# the CRW SST NetCDF. Defines sample_5x5(), which extracts
# the mean of a 5×5 pixel window around each survey point —
# used by both Figure 5a (ECOSTRESS) and Figure 5b (CRW).
# ============================================================

# --- Load bleaching survey data ---
df_bleach = pd.read_csv(BLEACHING_CSV_PATH)
df_bleach.dropna(subset=[LAT_COL, LON_COL, BLEACH_COL], inplace=True)

# --- Load and reproject ECOSTRESS LST raster ---
lst_raster = rioxarray.open_rasterio(ECO_PATH)
if lst_raster.rio.crs.to_string() != TARGET_CRS:
    lst_raster = lst_raster.rio.reproject(TARGET_CRS)
if 'band' in lst_raster.dims:
    lst_raster = lst_raster.squeeze('band', drop=True)

# --- Load and prepare CRW SST (temporal mean over study window) ---
crw_ds  = xr.open_dataset(CRW_PATH)
crw_ds  = crw_ds.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
crw_ds  = crw_ds.rio.write_crs("EPSG:4326")
if crw_ds.rio.crs.to_string() != TARGET_CRS:
    crw_ds = crw_ds.rio.reproject(TARGET_CRS)
sst_raster = crw_ds['CRW_SST'].sel(
    time=slice(CRW_TIME_START, CRW_TIME_END)
).mean(dim='time', skipna=True)


def sample_5x5(df, raster):
    """
    Sample the mean temperature from a 5×5 pixel window around each
    survey point in df, using the given raster as the temperature source.

    For each point, the nearest raster pixel is located and a 5×5 window
    (±2 pixels in each direction, clipped to raster bounds) is extracted.
    Nodata values and zeros are excluded before computing the window mean.
    Points whose entire window is masked receive NaN.

    Parameters
    ----------
    df     : pd.DataFrame  - Survey data with LAT_COL and LON_COL columns.
    raster : xr.DataArray  - 2-D raster reprojected to TARGET_CRS.

    Returns
    -------
    pd.DataFrame - Copy of df with a 'Temp_Value' column added.
                   Rows where sampling yielded NaN are dropped.
    """
    nodata_val  = raster.rio.nodata
    x_size      = len(raster.x)
    y_size      = len(raster.y)
    sampled     = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        lat, lon = row[LAT_COL], row[LON_COL]
        try:
            # Locate the nearest pixel by coordinate value, then get its integer index
            cx = raster.indexes['x'].get_loc(raster.sel(x=lon, method='nearest').x.item())
            cy = raster.indexes['y'].get_loc(raster.sel(y=lat, method='nearest').y.item())

            # Extract the 5×5 window, clamping to raster bounds
            window = raster.isel(
                x=slice(max(0, cx - 2), min(x_size - 1, cx + 2) + 1),
                y=slice(max(0, cy - 2), min(y_size - 1, cy + 2) + 1)
            )

            # Mask nodata and fill values (0)
            if nodata_val is not None:
                window = window.where(window != nodata_val)
            window = window.where(window > 0)

            sampled.append(window.mean().item())
        except KeyError:
            sampled.append(np.nan)  # Point outside raster bounds

    out = df.copy()
    out['Temp_Value'] = sampled
    out.dropna(subset=['Temp_Value', BLEACH_COL], inplace=True)
    return out


In [ ]:
# ============================================================
# Figure 5a: Max Bleaching vs ECOSTRESS LST Threshold
# ============================================================
# Sites are split into two groups by LST_THRESHOLD. A Mann-Whitney
# U test compares bleaching distributions between groups. The
# test statistic, p-value, and rank-biserial effect size are
# annotated on the boxplot.
# ============================================================

df_5a = sample_5x5(df_bleach.copy(), lst_raster)

bins   = [-np.inf, LST_THRESHOLD, np.inf]
labels = [f"\u2264 {LST_THRESHOLD}\u00b0C", f"> {LST_THRESHOLD}\u00b0C"]
df_5a['Stress_Group'] = pd.cut(df_5a['Temp_Value'], bins=bins, labels=labels, right=True)

# --- Mann-Whitney U test with rank-biserial effect size ---
group_low  = df_5a[df_5a['Stress_Group'] == labels[0]][BLEACH_COL]
group_high = df_5a[df_5a['Stress_Group'] == labels[1]][BLEACH_COL]
U, p_value = stats.mannwhitneyu(group_low, group_high, alternative='two-sided')
r_rb = 1 - (2 * U) / (len(group_low) * len(group_high))  # rank-biserial r
magnitude = "small" if abs(r_rb) < 0.3 else "medium" if abs(r_rb) < 0.5 else "large"

# Print results
print(f"Mann-Whitney U = {U:.1f}, p = {p_value:.4f}")
print(f"Rank-biserial r = {r_rb:.4f} ({magnitude})")

# --- Boxplot ---
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(10, 8))

sns.boxplot(data=df_5a, x='Stress_Group', y=BLEACH_COL, order=labels,
            palette="YlOrRd", hue='Stress_Group', legend=False, ax=ax)
sns.stripplot(data=df_5a, x='Stress_Group', y=BLEACH_COL, order=labels,
              color=".25", alpha=0.3, ax=ax)

# n-count annotations
n_counts = df_5a['Stress_Group'].value_counts()
for i, label in enumerate(labels):
    grp  = df_5a[df_5a['Stress_Group'] == label][BLEACH_COL]
    y_pos = grp.max() + 3 if not grp.empty else 5
    ax.text(i, y_pos, f"n = {n_counts.get(label, 0)}",
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Stat annotation box
stat_text = (f"Mann-Whitney U = {U:.1f}\n"
             f"p = {p_value:.4f}\n"
             f"rank-biserial r = {r_rb:.2f} ({magnitude})")
ax.text(0.98, 0.02, stat_text, transform=ax.transAxes,
        fontsize=16, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.8))

ax.set_xlabel(f"ECOSTRESS Mean LST (May–Nov '23) in 5×5 Window", fontsize=18)
ax.set_ylabel("Max Percent Bleached", fontsize=18)
ax.set_title(f"Distribution of Max Coral Bleaching vs. LST Threshold ({LST_THRESHOLD}\u00b0C)", fontsize=16)
ax.tick_params(axis='both', labelsize=16)
ax.set_ylim(-5, 105)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Figure 5b: Max Bleaching vs CRW SST Threshold
# ============================================================
# Same threshold-boxplot approach as Figure 5a, but using the
# NOAA CRW mean SST composite as the temperature source instead
# of ECOSTRESS LST.
# ============================================================

df_5b = sample_5x5(df_bleach.copy(), sst_raster)

bins   = [-np.inf, LST_THRESHOLD, np.inf]
labels = [f"\u2264 {LST_THRESHOLD}\u00b0C", f"> {LST_THRESHOLD}\u00b0C"]
df_5b['Stress_Group'] = pd.cut(df_5b['Temp_Value'], bins=bins, labels=labels, right=True)

# --- Mann-Whitney U test ---
group_low  = df_5b[df_5b['Stress_Group'] == labels[0]][BLEACH_COL]
group_high = df_5b[df_5b['Stress_Group'] == labels[1]][BLEACH_COL]
U, p_value = stats.mannwhitneyu(group_low, group_high, alternative='two-sided')

print(f"Mann-Whitney U = {U:.1f}, p = {p_value:.4f}")

# --- Boxplot ---
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(10, 8))

sns.boxplot(data=df_5b, x='Stress_Group', y=BLEACH_COL, order=labels,
            palette="YlOrRd", hue='Stress_Group', legend=False,
            dodge=False, ax=ax)
sns.stripplot(data=df_5b, x='Stress_Group', y=BLEACH_COL, order=labels,
              color=".25", alpha=0.4, jitter=0.2, dodge=False, ax=ax)

# n-count annotations
n_counts = df_5b['Stress_Group'].value_counts()
for i, label in enumerate(labels):
    grp   = df_5b[df_5b['Stress_Group'] == label][BLEACH_COL]
    y_pos = grp.max() + 3 if not grp.empty else 5
    ax.text(i, y_pos, f"n = {n_counts.get(label, 0)}",
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel(f"CRW Mean SST (May–Nov '23) in 5×5 Window", fontsize=18)
ax.set_ylabel("Max Percent Bleached", fontsize=18)
ax.set_title(f"Distribution of Max Coral Bleaching vs. SST Threshold ({LST_THRESHOLD}\u00b0C)", fontsize=16)
ax.tick_params(axis='both', labelsize=16)
ax.set_ylim(-5, 110)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Figure 5c: Survey Site Map Coloured by LST Stress Group
# ============================================================
# Plots the ECOSTRESS LST composite as a background raster
# (magma colormap, fixed 27–34°C scale) and overlays survey
# sites coloured by their stress group assignment from Figure 5a:
#   cyan  = sites ≤ LST_THRESHOLD
#   green = sites > LST_THRESHOLD
# Requires df_5a and lst_raster from the preceding cells.
# ============================================================

df_low_stress  = df_5a[df_5a['Stress_Group'] == f"\u2264 {LST_THRESHOLD}\u00b0C"].copy()
df_high_stress = df_5a[df_5a['Stress_Group'] == f"> {LST_THRESHOLD}\u00b0C"].copy()

fig, ax = plt.subplots(figsize=(10, 8))

# Background: LST raster with fixed temperature colour scale
img = lst_raster.plot(
    ax=ax, cmap='magma', vmin=27.0, vmax=34.0, robust=False,
    cbar_kwargs={'label': "Mean LST (\u00b0C) (May\u2013Nov '23)", 'extend': 'both'}
)
img.colorbar.ax.tick_params(labelsize=14)
img.colorbar.set_label("Mean LST (\u00b0C) (May\u2013Nov '23)", fontsize=18)

# Survey points coloured by stress group
sns.scatterplot(data=df_low_stress,  x=LON_COL, y=LAT_COL, ax=ax,
                color='cyan',   edgecolor='black', s=40, alpha=0.7, zorder=4,
                label=f"Sites \u2264 {LST_THRESHOLD}\u00b0C (n={len(df_low_stress)})")
sns.scatterplot(data=df_high_stress, x=LON_COL, y=LAT_COL, ax=ax,
                color='#00FF00', edgecolor='black', s=80, linewidth=1.5, zorder=5,
                label=f"Sites > {LST_THRESHOLD}\u00b0C (n={len(df_high_stress)})")

ax.set_facecolor('#000000')  # Dark background for no-data ocean areas
ax.set_title('')
ax.set_xlabel('Longitude', fontsize=18)
ax.set_ylabel('Latitude',  fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.legend(facecolor='white', framealpha=0.8, loc='upper right', fontsize=14)
plt.tight_layout()
plt.show()
